In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    f1_score,
    classification_report
)

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.neighbors import NearestNeighbors

import lightgbm as lgb
import networkx as nx

# ============================================================
# 1. LOAD ASTRAM DATASET
# ============================================================

DATA_PATH="/content/Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv"

df=pd.read_csv(DATA_PATH)

df.columns=df.columns.str.lower()

print("Dataset shape:", df.shape)

display(df.head())


# ============================================================
# 2. PREPROCESSING
# ============================================================

for c in [
    "start_datetime",
    "closed_datetime"
]:
    if c in df.columns:
        df[c]=pd.to_datetime(
            df[c],
            errors="coerce"
        )

# time features
df["hour"] = df["start_datetime"].dt.hour.fillna(0)

df["dayofweek"] = df["start_datetime"].dt.dayofweek.fillna(0)

df["peak_window"] = (df['hour'].isin(range(4, 8)) | df['hour'].isin(range(19, 24))).astype(int)


# ============================================================
# MODULE A
# CODE MIXED NLP FEATURE EXTRACTION
# ============================================================

df["description"] = df["description"].fillna("").astype(str)

tfidf=TfidfVectorizer(
    max_features=200,
    analyzer="word",
    ngram_range=(1,2)
)

text_vectors=tfidf.fit_transform(
    df.description
)

text_df=pd.DataFrame(
    text_vectors.toarray(),
    columns=[
        f"text_{i}"
        for i in range(
            text_vectors.shape[1]
        )
    ]
)

df=pd.concat(
    [
        df.reset_index(drop=True),
        text_df
    ],
    axis=1
)

print("NLP Module Ready:", text_df.shape)



# ============================================================
# MODULE B
# SURVIVAL STYLE CLEARANCE TIME MODEL
# ============================================================

df["duration_hours"]=(df.closed_datetime - df.start_datetime).dt.total_seconds()/3600

# right censor approximation
df["event_observed"]=(df.duration_hours.notna()).astype(int)

median_duration=(df.duration_hours.median())

df["duration_hours"]=(df.duration_hours.fillna(median_duration).clip(0,200))




# ============================================================
# FEATURE ENCODING
# ============================================================

categorical_cols=[
"event_type",
"event_cause",
"corridor",
"junction",
"zone",
"gba_identifier",
"police_station",
"priority",
"veh_type"
]

encoders={}

for col in categorical_cols:
    if col in df.columns:
        df[col]=(
            df[col]
            .fillna("unknown")
            .astype(str)
        )

        le=LabelEncoder()
        df[col]=le.fit_transform(
            df[col]
        )
        encoders[col]=le

for col in df.columns:
    if df[col].dtype=="object":
        df[col]=0

# Exclude datetime columns from the general fillna(-1) to prevent type conversion issues
datetime_cols_to_preserve_nat = ["start_datetime", "closed_datetime"]

for col in df.columns:
    if col not in datetime_cols_to_preserve_nat:
        df[col] = df[col].fillna(-1)


features=[
"event_type",
"event_cause",
"latitude",
"longitude",
"endlatitude",
"endlongitude",
"corridor",
"junction",
"zone",
"gba_identifier",
"police_station",
"priority",
"veh_type",
"hour",
"dayofweek",
"peak_window"
]

features=[f for f in features if f in df.columns]

features += list(text_df.columns)

X=df[features]




# ============================================================
# TRAIN DURATION MODEL
# ============================================================

y=df.duration_hours

Xtrain,Xtest,ytrain,ytest=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

duration_model=lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=40
)

duration_model.fit(
    Xtrain,
    ytrain
)

pred=duration_model.predict(
    Xtest
)

print("\n==============================")
print("Duration Prediction Metrics")
print("==============================")

print("MAE:", mean_absolute_error(ytest, pred))

print("RMSE:", np.sqrt(mean_squared_error(ytest, pred)))

print("R2:", r2_score(ytest, pred))




# ============================================================
# MODULE C
# SPATIO TEMPORAL GRAPH MODEL
# ============================================================

G=nx.Graph()

if "junction" in df.columns:
    for j in df.junction.unique():
        G.add_node(j)

    ordered=df.sort_values(
        "start_datetime"
    )

    for i in range(
        len(ordered)-1
    ):
        a=ordered.iloc[i].junction
        b=ordered.iloc[i+1].junction

        if a!=b:
            if G.has_edge(a,b):
                G[a][b]["weight"]+=1
            else:
                G.add_edge(
                    a,
                    b,
                    weight=1
                )

print("\nGraph Nodes:", G.number_of_nodes())
print("Graph Edges:", G.number_of_edges())





# ============================================================
# MODULE F
# BARRICADE / CLOSURE CLASSIFIER
# ============================================================

if "requires_road_closure" in df.columns:
    y=df.requires_road_closure.astype(int)
else:
    y=(
        (df.duration_hours>5)
        |
        (df.priority==1)
    ).astype(int)

Xtr,Xte,ytr,yte=train_test_split(
    X,
    y,
    test_size=.2,
    random_state=10
)

closure_model=lgb.LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05
)

closure_model.fit(
    Xtr,
    ytr
)

ypred=closure_model.predict(
    Xte
)

print("\n==============================")
print("Closure Classification Metrics")
print("==============================")

print("Accuracy:", accuracy_score(yte, ypred))

print("F1:", f1_score(yte, ypred))

print(classification_report(yte, ypred))






# ============================================================
# MODULE H
# CASE MEMORY RETRIEVAL ENGINE
# ============================================================

retriever=NearestNeighbors(
    n_neighbors=5,
    metric="cosine"
)

retriever.fit(X)

dist,idx = retriever.kneighbors(
    X.iloc[:5]
)

print("\nAverage Retrieval Similarity:", 1-np.mean(dist))






# ============================================================
# MODULE G
# MANPOWER MODEL
# ============================================================

def manpower_predict(
    duration,
    closure,
    priority
):
    manpower=5
    manpower += duration*1.5

    if priority==1:
        manpower+=15

    if closure:
        manpower+=10

    manpower=int(manpower)

    return {
    "total_headcount": manpower,
    "roles":{
        "traffic_constable": int(manpower*.45),
        "home_guard": int(manpower*.30),
        "civil_defense": int(manpower*.15),
        "barricade_team": int(manpower*.10)
    },
    "90_percent_range":
        [
            max(1,manpower-5),
            manpower+5
        ]
    }





# ============================================================
# FINAL PREDICTION PIPELINE
# ============================================================

def predict_event(sample):
    x=sample[features].values.reshape(1,-1)

    duration=float(
        duration_model.predict(x)[0]
    )

    closure=int(
        closure_model.predict(x)[0]
    )

    result={
    "duration_hours": duration,
    "barricade_action":
        "full_barricade"
        if closure
        else
        "partial_barricade"
        if duration>3
        else
        "cones",
    "manpower_required":
        manpower_predict(
            duration,
            closure,
            sample.priority
        )
    }

    return result





# ============================================================
# RANDOM REAL EVENT TEST
# ============================================================

print("\n==============================")
print("Random Event Test")
print("==============================")

sample=df.sample(1).iloc[0]

print(predict_event(sample))




# ============================================================
# SAVE COMPLETE ML ENGINE
# ============================================================

engine={
"duration_model": duration_model,
"closure_model": closure_model,
"text_encoder": tfidf,
"label_encoders": encoders,
"graph": G,
"retriever": retriever,
"features": features,
"version": "ASTRAM Event Congestion Engine v1"
}

SAVE_PATH="/content/event_congestion_engine.pkl"

joblib.dump(
    engine,
    SAVE_PATH
)

print("\nMODEL SAVED AT:", SAVE_PATH )
print("SIZE MB:", os.path.getsize(SAVE_PATH)/1024/1024)


Dataset shape: (8173, 46)


,id,event_type,latitude,longitude,endlatitude,endlongitude,address,end_address,event_cause,requires_road_closure,...,resolved_at_address,resolved_at_latitude,resolved_at_longitude,closed_by_id,closed_datetime,resolved_by_id,resolved_datetime,gba_identifier,zone,junction
0,FKID000000,unplanned,13.040004,77.518099,0.000000,0.000000,"Mumbai Bengaluru Highway, Jalahalli Cross Junc...",NaN,vehicle_breakdown,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FKID000001,unplanned,12.921876,77.645158,0.000000,0.000000,"19th Main Road, Heavie Halcyon, Agara, HSR Lay...",NaN,vehicle_breakdown,False,...,"19th Main Road, Heavie Halcyon, Agara, HSR Lay...",12.921876,77.645158,NaN,NaN,FKUSR00002,2024-01-30 04:17:46.828355+00,NaN,NaN,NaN
2,FKID000002,unplanned,12.955622,77.585708,0.000000,0.000000,"Lalbagh Main Road, Dr Sri Shantaveera Swami Ci...",NaN,others,False,...,NaN,NaN,NaN,FKUSR00003,2024-01-30 04:56:03.281509+00,NaN,NaN,Bengaluru Central Corporation,Central Zone 2,UrvashiJunction
3,FKID000003,unplanned,13.006147,77.579435,13.006239,77.579516,"Sankey Road, Bashyam Circle, Sadashiva Nagar, ...","Sankey Road, Palace Orchard Upper, Sadashiva N...",tree_fall,True,...,NaN,NaN,NaN,FKUSR00004,2024-03-14 07:42:05.54944+00,NaN,NaN,NaN,NaN,NaN
4,FKID000004,unplanned,12.953980,77.585233,0.000000,0.000000,"Lalbagh Fort Road, Lalbagh Main Gate Junction,...",NaN,vehicle_breakdown,False,...,NaN,NaN,NaN,FKUSR00003,2024-01-30 05:35:17.338283+00,NaN,NaN,NaN,NaN,LalbaghMainGateJunc


NLP Module Ready: (8173, 200)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008618 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 11564
[LightGBM] [Info] Number of data points in the train set: 6538, number of used features: 216
[LightGBM] [Info] Start training from score 13.363987

Duration Prediction Metrics
MAE: 18.338871057815886
RMSE: 40.876097115491895
R2: 0.2627648873027355

Graph Nodes: 295
Graph Edges: 1583
[LightGBM] [Info] Number of positive: 536, number of negative: 6002
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012969 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 11437
[LightGBM] [Info] Number of data points in the train set: 6538, number of used features: 216
[Li